# Retriever Debug
Compare semantic, BM25, and hybrid retrieval side-by-side.
Change `QUERY` and re-run the comparison cell to debug.

In [1]:
from pathlib import Path
import sys

repo_root = next(
    (p for p in [Path.cwd(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").exists()),
    Path.cwd().resolve(),
)
sys.path.insert(0, str(repo_root))

from app.retriever import (
    BM25Config,
    SemanticConfig,
    get_bm25_retriever,
    get_hybrid_retriever,
    get_semantic_retriever,
)

print("Imports OK")

/home/mahee/Work/Thesis/Repos/langchain-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


In [2]:
# ── Build retrievers (run once — BM25 loads all docs from Chroma at init) ────
semantic = get_semantic_retriever(SemanticConfig(k=6))
bm25     = get_bm25_retriever(BM25Config(k=6))
hybrid   = get_hybrid_retriever([
    (get_semantic_retriever(SemanticConfig(k=6)), 0.5),
    (get_bm25_retriever(BM25Config(k=6)),         0.5),
])

print("Retrievers ready")

Retrievers ready


In [9]:
# ── Change this to test different queries ─────────────────────────────────────
# QUERY = "What is NISQ"
# QUERY = "What is Noisy Intermediate Scale Quantum (NISQ) computing?"  # longer query to test BM25 recall
QUERY = "MLflow Tracking API data capture for experiment tracking and analysis in quantum software development."

In [7]:
# ── Inspect raw similarity scores from ChromaDB ───────────────────────────────
# Run this to find the right score_threshold value for SemanticConfig.
# Look at the gap between relevant and irrelevant docs — set threshold just above
# the irrelevant cluster.

from app.vectorstore import get_vectorstore

vs = get_vectorstore()
results = vs.similarity_search_with_relevance_scores(QUERY, k=10)

print(f"Query: {QUERY!r}\n")
print(f"{'Score':>6}  {'Corpus':<10}  Title / section")
print("─" * 80)
for doc, score in results:
    m       = doc.metadata
    corpus  = m.get("source_corpus", "?")
    title   = (m.get("title") or m.get("source_file", "?").split("/")[-1])[:55]
    section = (m.get("section") or "")[:40]
    label   = f"{title}  {section}".strip()
    print(f"  {score:.4f}  [{corpus:<8}]  {label}")

Query: 'MLflow Tracking API data capture for experiment tracking and analysis'

 Score  Corpus      Title / section
────────────────────────────────────────────────────────────────────────────────
  0.7319  [mlflow  ]  index  MLflow Tracking APIs > Choose Your Appro
  0.7002  [unknown ]  Kinanen et al. - 2025 - Toolchain for Experiment Tracki  Toolchain for experiment tracking in ite
  0.6746  [mlflow  ]  Fine-Tuning Transformers with MLflow for Enhanced Model  Integrating MLflow for Experiment Tracki
  0.6707  [mlflow  ]  index  Remote Experiment Tracking with MLflow T
  0.6687  [mlflow  ]  index  MLflow Tracking APIs
  0.6532  [mlflow  ]  index  MLflow Tracking > Concepts > Experiments
  0.6526  [unknown ]  Kinanen et al. - 2025 - Toolchain for Experiment Tracki  Toolchain for experiment tracking in ite
  0.6451  [mlflow  ]  index  Getting Started with MLflow > Experiment
  0.6439  [mlflow  ]  Logging Visualizations with MLflow  Detailed Overview of Main Execution for
  0.6375  [mlfl

In [10]:
# ── Side-by-side comparison ───────────────────────────────────────────────────
def show_results(label: str, docs: list, snippet_len: int = 200) -> None:
    print(f"\n{'─'*60}")
    print(f"  {label}  ({len(docs)} results)")
    print(f"{'─'*60}")
    for i, doc in enumerate(docs, 1):
        m = doc.metadata
        corpus  = m.get("source_corpus", "?")
        title   = m.get("title") or m.get("source_file", "?").split("/")[-1]
        section = m.get("section") or m.get("h2") or ""
        snippet = doc.page_content[:snippet_len].replace("\n", " ")
        print(f"  {i}. [{corpus}] {title}")
        if section:
            print(f"     section : {section}")
        print(f"     snippet : {snippet!r}")


print(f"Query: {QUERY!r}")
show_results("SEMANTIC", semantic.invoke(QUERY))
show_results("BM25    ", bm25.invoke(QUERY))
show_results("HYBRID  ", hybrid.invoke(QUERY))

Query: 'MLflow Tracking API data capture for experiment tracking and analysis in quantum software development.'

────────────────────────────────────────────────────────────
  SEMANTIC  (6 results)
────────────────────────────────────────────────────────────
  1. [unknown] Gamage et al. - 2025 - Enhancing Quantum Software Development Process with Experiment Tracking
     section : Enhancing Quantum Software Development Process with Experiment Tracking > II. BACKGROUND AND MOTIVATION
     snippet : '## II. BACKGROUND AND MOTIVATION   Experiment tracking [7] was introduced as a tool that addresses the challenges faced by machine learning (ML) developers in the following four areas: multitude of to'
  2. [unknown] Kinanen et al. - 2025 - Toolchain for Experiment Tracking in Iterative Quantum Software Development
     section : Toolchain for experiment tracking in iterative quantum software development
     snippet : '# Toolchain for experiment tracking in iterative quantum software develo

In [8]:
# ── Inspect full metadata for a single result ─────────────────────────────────
# Change retriever and index to inspect any result in detail
RETRIEVER = hybrid
INDEX = 0  # 0-based

docs = RETRIEVER.invoke(QUERY)
doc  = docs[INDEX]

print("=== Metadata ===")
for k, v in doc.metadata.items():
    print(f"  {k}: {v}")

print("\n=== Full content ===")
print(doc.page_content)

=== Metadata ===
  content_type: narrative
  source_file: knowledge_ingestion/content/v3/content/tech_docs/mlflow/docs/docs/classic-ml/deployment/deploy-model-to-kubernetes/tutorial.mdx
  section: Develop ML model with MLflow and deploy to Kubernetes > Introduction: Scalable Model Serving with KServe and MLServer > What is KServe?
  title: tutorial
  h2: Introduction: Scalable Model Serving with KServe and MLServer
  h1: Develop ML model with MLflow and deploy to Kubernetes
  format: mdx
  h3: What is KServe?
  description: 
  source_corpus: mlflow

=== Full content ===
### What is KServe?  
[KServe](https://kserve.github.io/website), formally known as KFServing, provides performant, scalable, and highly-abstracted interfaces for common machine learning frameworks like Tensorflow, XGBoost, scikit-learn, and Pytorch.
It offers advanced features that aid in operating large-scale machine learning systems, such as **autoscaling**, **canary rollout**, **A/B testing**, **monitoring**,
**expl

In [9]:
# ── Batch test multiple queries ───────────────────────────────────────────────
# Useful for a quick sanity check across several terms at once
TEST_QUERIES = [
    "NISQ",
    "VQE",
    "QAOA",
    "QProv",
    "quantum circuit",
    "experiment tracking",
]

RETRIEVER = hybrid
SHOW_TOP_N = 3

for q in TEST_QUERIES:
    results = RETRIEVER.invoke(q)
    corpora = [d.metadata.get("source_corpus", "?") for d in results[:SHOW_TOP_N]]
    titles  = [d.metadata.get("title") or "?" for d in results[:SHOW_TOP_N]]
    print(f"\n{q!r:40s}  top-{SHOW_TOP_N} corpora: {corpora}")
    for c, t in zip(corpora, titles):
        print(f"    [{c}] {t[:80]}")


'NISQ'                                    top-3 corpora: ['mlflow', 'unknown', 'mlflow']
    [mlflow] Fine-Tuning Open-Source LLM using QLoRA with MLflow and PEFT
    [unknown] Preskill - 2018 - Quantum Computing in the NISQ era and beyond
    [mlflow] Deploy an MLflow `PyFunc` model with Model Serving

'VQE'                                     top-3 corpora: ['mlflow', 'unknown', 'mlflow']
    [mlflow] Fine-Tuning Open-Source LLM using QLoRA with MLflow and PEFT
    [unknown] Preskill - 2018 - Quantum Computing in the NISQ era and beyond
    [mlflow] Deploy an MLflow `PyFunc` model with Model Serving

'QAOA'                                    top-3 corpora: ['mlflow', 'unknown', 'mlflow']
    [mlflow] Fine-Tuning Open-Source LLM using QLoRA with MLflow and PEFT
    [unknown] Preskill - 2018 - Quantum Computing in the NISQ era and beyond
    [mlflow] Deploy an MLflow `PyFunc` model with Model Serving

'QProv'                                   top-3 corpora: ['mlflow', 'qiskit', 'mlflo